# Genome Assembler: Colab Demo
Builds the C++ assembler from GitHub, simulates reads from a phage genome, runs the assembler, and evaluates the contigs.

**Before running:** push your repo to GitHub and set `REPO_URL` in cell 2.
Runtime tip: *Runtime > Run all*. Nothing here needs a GPU.

## 1. Setup

In [ ]:
# Edit this to your repo
REPO_URL = "https://github.com/YOUR_USERNAME/genome-assembler.git"
REPO_DIR = "genome-assembler"

# Optional: keep large outputs on Google Drive so they survive runtime resets
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/genome-assembler-results"
else:
    OUT_DIR = "/content/results"

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("Outputs go to:", OUT_DIR)

In [ ]:
# Fresh clone (or pull if it already exists)
import os
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q matplotlib

## 2. Build the C++ assembler

In [ ]:
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build -j4

## 3. Run unit tests

In [ ]:
!ctest --test-dir build --output-on-failure

## 4. Get a small genome
phiX174 (~5.4 kb) is perfect for development. Swap the accession for lambda phage (`NC_001416.1`, ~48 kb) once things work.

In [ ]:
GENOME_ID = "NC_001422.1"   # phiX174
GENOME_FA = f"data/{GENOME_ID}.fa"
!wget -q -O {GENOME_FA} "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={GENOME_ID}&rettype=fasta"
!head -c 200 {GENOME_FA}; echo
!grep -v ">" {GENOME_FA} | tr -d '\n' | wc -c

## 5. Simulate reads

In [ ]:
READS = f"{OUT_DIR}/reads.fastq"
!python python/simulate_reads.py {GENOME_FA} -o {READS} --read-len 100 --coverage 50 --error 0.01 --seed 42
!head -8 {READS}

## 6. Run the assembler

In [ ]:
K = 31
CONTIGS = f"{OUT_DIR}/contigs.fasta"
!./build/assembler {READS} -k {K} -o {CONTIGS}

## 7. Evaluate the contigs
Works once your assembler writes `contigs.fasta`. N50 = the contig length such that contigs of that length or longer cover at least half of the assembly.

In [ ]:
import os

def read_fasta_lengths(path):
    lengths, cur = [], 0
    with open(path) as f:
        for line in f:
            if line.startswith(">"):
                if cur: lengths.append(cur)
                cur = 0
            else:
                cur += len(line.strip())
    if cur: lengths.append(cur)
    return lengths

def n50(lengths):
    total, running = sum(lengths), 0
    for l in sorted(lengths, reverse=True):
        running += l
        if running >= total / 2:
            return l
    return 0

if os.path.exists(CONTIGS):
    lens = read_fasta_lengths(CONTIGS)
    if lens:
        print(f"Contigs: {len(lens)} | Total: {sum(lens)} bp | Longest: {max(lens)} bp | N50: {n50(lens)}")
    else:
        print("contigs.fasta is empty (assembler not implemented yet).")
else:
    print("No contigs file yet; implement the pipeline first.")

In [ ]:
import matplotlib.pyplot as plt

if os.path.exists(CONTIGS) and read_fasta_lengths(CONTIGS):
    lens = sorted(read_fasta_lengths(CONTIGS), reverse=True)
    plt.bar(range(len(lens)), lens)
    plt.xlabel("Contig rank"); plt.ylabel("Length (bp)")
    plt.title(f"Contig lengths (k={K})")
    plt.show()

## 8. Compare with a real assembler (optional, later)
Once your assembler works, install `quast` (`!pip install quast`) and compare against the reference:
`!quast contigs.fasta -r data/NC_001422.1.fa -o quast_out`
You can also run SPAdes or MEGAHIT on the same reads for a benchmark table in your README.